# Day 084 — Exercise 2: The Writer Agent

**What you'll build:** `build_writer_prompt` and `WriterAgent` — a specialist that takes findings (from the researcher or any other source) and produces a polished document.

**Why it matters:** the writer is the second half of the collaboration. It receives findings as a plain string and has full freedom to shape the output — the style (concise, detailed, formal) is set at construction time so the same agent can serve different audiences. The clean interface — `write(findings) -> str` — mirrors the researcher's.

In [ ]:
def _mock_multi(findings='Finding: AI agents collaborate.', document='Doc: Agents work together.'):
    """Branch on system message: 'research specialist' -> findings, else -> document."""
    def _fn(messages):
        system = messages[0]['content'] if messages else ''
        if 'research specialist' in system.lower():
            return findings
        return document
    return _fn
import json

# ── helpers reused from Day 79 ───────────────────────────────────────────────
def safe_parse_json(text):
    """Slice first '{' to last '}' and parse. Returns dict|None (Day 79)."""
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1 or end < start:
        return None
    try:
        data = json.loads(text[start:end + 1])
    except (json.JSONDecodeError, ValueError):
        return None
    return data if isinstance(data, dict) else None


def call_llm(messages, llm_fn=None):
    """Call the chat model, or the injected llm_fn(messages) -> str (Day 79)."""
    if llm_fn is not None:
        return llm_fn(messages)
    import ollama
    resp = ollama.chat(model="llama3.2", messages=messages)
    return resp["message"]["content"]

# ── researcher specialist ─────────────────────────────────────────────────────
def build_researcher_prompt(query):
    """Build a prompt for the researcher role: gather facts on a topic."""
    system = "\n".join([
        "You are a research specialist. Your job is to gather relevant facts and",
        "key information about the topic given to you.",
        "",
        "Return a structured list of the most important findings.",
        "Be factual, concise, and cover the main points.",
    ])
    return [{"role": "system", "content": system},
            {"role": "user", "content": "Research topic: " + str(query)}]


class ResearcherAgent:
    """A specialist that researches a topic and returns structured findings.

    Each call to research() returns a string of findings and records the
    exchange in history. The agent has one job: gather facts. It passes its
    output to the next agent via a Handoff — it does not write, review, or
    plan.

    Example::

        researcher = ResearcherAgent(llm_fn=my_llm_fn)
        findings = researcher.research("topological sort algorithms")
    """

    def __init__(self, llm_fn=None):
        self._llm_fn = llm_fn
        self._history = []

    def research(self, query):
        """Research a query and return findings as a string."""
        messages = build_researcher_prompt(query)
        findings = call_llm(messages, llm_fn=self._llm_fn)
        self._history.append({"query": query, "findings": findings})
        return findings

    def history(self):
        """Return a copy of the research history."""
        return list(self._history)

    def clear_history(self):
        """Clear the history in place."""
        self._history.clear()


## Task

1. `build_writer_prompt(findings, style='concise', instructions=None)` — a `system` message orienting the agent as a writing specialist with the given style; append `instructions` to the system if provided; a `user` message with the findings.
2. `WriterAgent(llm_fn=None, style='concise')` — `write(findings, instructions=None)` calls `call_llm` and records `{'findings', 'document'}` in `_history`; returns the document string. `history()` copy; `clear_history()` in place.

## Your Implementation

In [ ]:
def build_writer_prompt(findings, style='concise', instructions=None):
    """Build a prompt for the writer role: turn findings into a document."""
    raise NotImplementedError

class WriterAgent:
    """A specialist that turns research findings into a polished document."""

    def __init__(self, llm_fn=None, style='concise'):
        raise NotImplementedError

    def write(self, findings, instructions=None):
        raise NotImplementedError

    def history(self):
        raise NotImplementedError

    def clear_history(self):
        raise NotImplementedError


In [ ]:

# ── writer specialist ─────────────────────────────────────────────────────────
def build_writer_prompt(findings, style="concise", instructions=None):
    """Build a prompt for the writer role: turn findings into a document."""
    system_parts = [
        "You are a writing specialist. Turn the provided findings into",
        "a polished, well-structured document.",
        "Style: " + str(style) + ".",
    ]
    if instructions:
        system_parts.append("Additional instructions: " + str(instructions))
    system = "\n".join(system_parts)
    user = "Findings:\n" + str(findings)
    return [{"role": "system", "content": system},
            {"role": "user", "content": user}]


class WriterAgent:
    """A specialist that turns research findings into a polished document.

    The writer has one job: take findings (a string from the ResearcherAgent
    or any other source) and produce a well-structured document. The style
    controls the tone (e.g. 'concise', 'detailed', 'formal').

    Example::

        writer = WriterAgent(llm_fn=my_llm_fn, style="concise")
        document = writer.write(findings)
    """

    def __init__(self, llm_fn=None, style="concise"):
        self._llm_fn = llm_fn
        self.style = style
        self._history = []

    def write(self, findings, instructions=None):
        """Write a document from findings. Returns the document string."""
        messages = build_writer_prompt(findings, style=self.style,
                                       instructions=instructions)
        document = call_llm(messages, llm_fn=self._llm_fn)
        self._history.append({"findings": findings, "document": document})
        return document

    def history(self):
        """Return a copy of the writing history."""
        return list(self._history)

    def clear_history(self):
        """Clear the history in place."""
        self._history.clear()


## Automated checks

In [ ]:

score, total = 0, 5
try:
    msgs = build_writer_prompt('Fact: X.', style='formal')
    assert msgs[0]['role'] == 'system' and 'formal' in msgs[0]['content'].lower()
    assert 'Fact: X.' in msgs[1]['content']
    score += 1; print("✅ build_writer_prompt includes style and findings")

    w = WriterAgent(llm_fn=_mock_multi(document='The document.'), style='concise')
    doc = w.write('some findings')
    assert doc == 'The document.'
    score += 1; print("✅ write() returns the LLM output")

    assert len(w.history()) == 1 and 'findings' in w.history()[0]
    score += 1; print("✅ history records each write")

    w.history().clear()
    assert len(w.history()) == 1
    score += 1; print("✅ history() returns a copy")

    w.clear_history()
    assert len(w.history()) == 0
    score += 1; print("✅ clear_history() empties in place")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python

# ── writer specialist ─────────────────────────────────────────────────────────
def build_writer_prompt(findings, style="concise", instructions=None):
    """Build a prompt for the writer role: turn findings into a document."""
    system_parts = [
        "You are a writing specialist. Turn the provided findings into",
        "a polished, well-structured document.",
        "Style: " + str(style) + ".",
    ]
    if instructions:
        system_parts.append("Additional instructions: " + str(instructions))
    system = "\n".join(system_parts)
    user = "Findings:\n" + str(findings)
    return [{"role": "system", "content": system},
            {"role": "user", "content": user}]


class WriterAgent:
    """A specialist that turns research findings into a polished document.

    The writer has one job: take findings (a string from the ResearcherAgent
    or any other source) and produce a well-structured document. The style
    controls the tone (e.g. 'concise', 'detailed', 'formal').

    Example::

        writer = WriterAgent(llm_fn=my_llm_fn, style="concise")
        document = writer.write(findings)
    """

    def __init__(self, llm_fn=None, style="concise"):
        self._llm_fn = llm_fn
        self.style = style
        self._history = []

    def write(self, findings, instructions=None):
        """Write a document from findings. Returns the document string."""
        messages = build_writer_prompt(findings, style=self.style,
                                       instructions=instructions)
        document = call_llm(messages, llm_fn=self._llm_fn)
        self._history.append({"findings": findings, "document": document})
        return document

    def history(self):
        """Return a copy of the writing history."""
        return list(self._history)

    def clear_history(self):
        """Clear the history in place."""
        self._history.clear()
```

**Why set style at construction, not per call?** Style is a property of the agent's *role* in this pipeline — a formal-writing agent stays formal across all its calls. If you want a different style, create a new WriterAgent. Binding it at construction makes each agent's behavior predictable and stable.

</details>